# Polarization curve simulation

This notebook shows how to build a PEMFC model and compute a polarization curve
with **marapendi**.

The workflow is:
1. Assemble a `FuelCell` from its components
2. Create `OperatingConditions` for cathode and anode
3. Call `compute_ui_curve` to sweep current density
4. Plot cell voltage and high-frequency resistance (HFR)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import marapendi as mrpd

## 1. Build the cell

A `FuelCell` is assembled bottom-up:

```
FuelCell
 ├── ca : FuelCellSide
 │    ├── cl  : PtCCatalystLayer  (ORR kinetics, ionomer film, Pt loading)
 │    ├── gdl : GasDiffusionLayer (thickness, porosity, tortuosity)
 │    └── ch  : FlowChannel       (geometry, gas-transport model)
 ├── an : FuelCellSide
 │    └── ...  (same structure, HOR side)
 └── membrane : PFSA              (thickness, conductivity, water transport)
```

Static material/geometry parameters live in the component tree.
Runtime state (temperatures, compositions, fluxes) is stored in `FuelCell.state`.

In [ ]:
# ── Shared two-phase transport model ────────────────────────────────────────
liq = mrpd.DarcyTransportModel(J_function_exponent=2)

# ── ORR kinetics ─────────────────────────────────────────────────────────────
orr = mrpd.ElectrochemicalReaction(
    reference_exchange_current_density=2.5e-4,   # A/m²_Pt
    reaction_order=0.54,
    activation_energy=67e6,                       # J/kmol
    reference_activity=1e5,                        # Pa
    reference_temperature=353.15,
    number_of_electrons=2,
    charge_transfer_coeff=0.5,
)

# ── Cathode catalyst layer ────────────────────────────────────────────────────
cl_ca = mrpd.PtCCatalystLayer(
    ecsa=70e3,                         # m²/kg
    platinum_loading=0.4e-2,           # kg/m² (0.4 mg/cm²)
    ionomer=mrpd.PFSAIonomer(),
    reaction=orr,
    thickness=10e-6,
    thermal_conductivity=0.22,
    pore_diameter=40e-9,
    absolute_permeability=1e-13,
    contact_angle=97.,
    two_phase_transport_model=liq,
)

# ── GDL (same geometry for both sides) ────────────────────────────────────────
def make_gdl():
    return mrpd.GasDiffusionLayer(
        thickness=200e-6,
        porosity=0.6,
        contact_angle=120.,
        effective_gas_diffusion_ratio=0.3,
        absolute_permeability=1e-12,
        thermal_conductivity=0.5,
        two_phase_transport_model=liq,
    )

# ── Flow channels ─────────────────────────────────────────────────────────────
def make_channel(reactant):
    return mrpd.FlowChannel(
        width=1e-3, height=1e-3, length=0.1, n_parallel=20,
        reactant=reactant,
    )

# ── Assemble the cell ─────────────────────────────────────────────────────────
cell = mrpd.FuelCell(
    area=25e-4,                  # 25 cm²
    electrical_resistance=30e-7, # Ω·m²
    ca=mrpd.FuelCellSide(
        cl=cl_ca,
        gdl=make_gdl(),
        ch=make_channel('o2'),
        has_mpl=False,
        thermal_contact_resistance=4e-4,
    ),
    an=mrpd.FuelCellSide(
        cl=mrpd.PtCCatalystLayer(
            thickness=5e-6, two_phase_transport_model=liq,
        ),
        gdl=make_gdl(),
        ch=make_channel('h2'),
        has_mpl=False,
        thermal_contact_resistance=4e-4,
    ),
    membrane=mrpd.PFSA(
        equivalent_weight=1100,
        dry_density=1980,
        dry_thickness=25e-6,
        water_balance_model=mrpd.MembraneWaterBalanceModel(),
    ),
    use_eq_water_content_for_ionomer=True,
)

print("Cell assembled.")
print(f"  Cathode porous layers: {[type(l).__name__ for l in cell.ca.porous_layers]}")
print(f"  Membrane thickness   : {cell.membrane.dry_thickness*1e6:.0f} µm")

## 2. Define operating conditions

`OperatingConditions` groups the inlet gas state (temperature, pressure,
relative humidity, dry composition) and the flow stoichiometry.

In [ ]:
T = 353.15  # K

ca_cond = mrpd.OperatingConditions(
    inlet_temperature=T,
    inlet_pressure=1.5e5,
    outlet_pressure=1.5e5,
    dry_o2_mole_fraction=0.21,
    inlet_relative_humidity=0.5,
    stoichiometry=2.0,
)

an_cond = mrpd.OperatingConditions(
    inlet_temperature=T,
    inlet_pressure=1.5e5,
    outlet_pressure=1.5e5,
    dry_h2_mole_fraction=1.0,
    inlet_relative_humidity=0.5,
    stoichiometry=1.5,
)

## 3. Compute the polarization curve

`compute_ui_curve` accepts a NumPy array of current densities (A/m²) and returns
the corresponding cell voltages. Call it with a single point to also read the HFR
at that operating point.

In [ ]:
i_Acm2 = np.array([0.1, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0])
i_Am2  = i_Acm2 * 1e4   # A/m²

voltages = []
hfr_vals = []

for i_k in i_Am2:
    V = cell.compute_ui_curve(np.array([i_k]), T, ca_cond, an_cond)
    voltages.append(float(np.atleast_1d(V)[0]))
    hfr_vals.append(float(np.atleast_1d(cell.high_frequency_resistance())[0]))

print(f"{'i (A/cm²)':>12}  {'V (V)':>8}  {'HFR (mΩ·cm²)':>14}")
print("-" * 40)
for i, V, R in zip(i_Acm2, voltages, hfr_vals):
    print(f"{i:12.1f}  {V:8.4f}  {R*1e3*1e4:14.1f}")

## 4. Plot

In [ ]:
fig, ax1 = plt.subplots(figsize=(6, 4))

ax1.plot(i_Acm2, voltages, 'C0o-', label='Cell voltage')
ax1.set_xlabel('Current density (A/cm²)')
ax1.set_ylabel('Cell voltage (V)', color='C0')
ax1.set_ylim(0, 1.1)
ax1.tick_params(axis='y', labelcolor='C0')

ax2 = ax1.twinx()
ax2.plot(i_Acm2, [r * 1e3 * 1e4 for r in hfr_vals], 'C1s--', label='HFR')
ax2.set_ylabel('HFR (mΩ·cm²)', color='C1')
ax2.tick_params(axis='y', labelcolor='C1')

fig.suptitle('PEMFC polarization curve — marapendi')
fig.tight_layout()
plt.show()

## 5. Sensitivity to operating conditions

Compare two RH levels to show how water management affects performance.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

for rh, color in [(0.3, 'C0'), (0.5, 'C1'), (0.8, 'C2')]:
    ca = mrpd.OperatingConditions(
        inlet_temperature=T, inlet_pressure=1.5e5, outlet_pressure=1.5e5,
        dry_o2_mole_fraction=0.21, inlet_relative_humidity=rh, stoichiometry=2.0,
    )
    an = mrpd.OperatingConditions(
        inlet_temperature=T, inlet_pressure=1.5e5, outlet_pressure=1.5e5,
        dry_h2_mole_fraction=1.0, inlet_relative_humidity=rh, stoichiometry=1.5,
    )
    vv = []
    for i_k in i_Am2:
        V = cell.compute_ui_curve(np.array([i_k]), T, ca, an)
        vv.append(float(np.atleast_1d(V)[0]))
    ax.plot(i_Acm2, vv, color=color, marker='o', label=f'RH = {int(rh*100)} %')

ax.set_xlabel('Current density (A/cm²)')
ax.set_ylabel('Cell voltage (V)')
ax.set_title('Effect of inlet relative humidity')
ax.legend()
ax.set_ylim(0, 1.1)
fig.tight_layout()
plt.show()